In [1]:
!pip install unsloth
!pip install snac
!pip install torchaudio
!pip install datasets transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 72.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 846.3 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive/rio"
os.makedirs(DRIVE, exist_ok=True)
print("Drive folder:", DRIVE)
print("Contents:", os.listdir(DRIVE))

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/rio
Contents: ['dataset.zip']


In [3]:
import zipfile, os, json

DRIVE    = "/content/drive/MyDrive/rio"
ZIP_PATH = f"{DRIVE}/dataset.zip"
WORK_DIR = "/content/work"
WAV_DIR  = f"{WORK_DIR}/wavs"

os.makedirs(WORK_DIR, exist_ok=True)

print("Extracting", ZIP_PATH, "...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)

# Find dataset.jsonl — may be at root or in a subfolder
jsonl_path = None
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == "dataset.jsonl":
            jsonl_path = os.path.join(root, f)
            break

# Find wavs dir
wavs_path = None
for root, dirs, files in os.walk(WORK_DIR):
    for d in dirs:
        if d == "wavs":
            wavs_path = os.path.join(root, d)
            break
    if wavs_path:
        break

if wavs_path is None:
    # Flat zip — wavs directly in WORK_DIR
    wavs_path = WORK_DIR

print(f"WAVs at: {wavs_path}  ({len(os.listdir(wavs_path))} files)")

if jsonl_path:
    print(f"dataset.jsonl at: {jsonl_path}")
    with open(jsonl_path) as f:
        samples = [json.loads(l) for l in f if l.strip()]
    print(f"Transcriptions: {len(samples)}")
else:
    print("WARNING: dataset.jsonl not found inside zip")
    print("  Place it at MyDrive/rio/dataset.jsonl manually")
    jsonl_path = f"{DRIVE}/dataset.jsonl"

Extracting /content/drive/MyDrive/rio/dataset.zip ...
WAVs at: /content/work/dataset/wavs  (136 files)
dataset.jsonl at: /content/work/dataset/dataset.jsonl
Transcriptions: 135


In [4]:
import json, os, torch, torchaudio
from snac import SNAC
from pathlib import Path
from tqdm import tqdm

DRIVE       = "/content/drive/MyDrive/rio"
WORK_DIR    = "/content/work"
MAX_AUDIO_S = 8   # skip clips longer than this

device = "cuda" if torch.cuda.is_available() else "cpu"
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(device)

def audio_to_tokens(wav_path):
    waveform, sr = torchaudio.load(wav_path)
    if sr != 24000:
        waveform = torchaudio.functional.resample(waveform, sr, 24000)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if waveform.shape[-1] / 24000 > MAX_AUDIO_S:
        return None
    with torch.no_grad():
        codes = snac_model.encode(waveform.unsqueeze(0).to(device))
    c0 = codes[0].squeeze().cpu().tolist()
    c1 = codes[1].squeeze().cpu().tolist()
    c2 = codes[2].squeeze().cpu().tolist()
    if len(c1) < 2 * len(c0) or len(c2) < 4 * len(c0):
        return None
    tokens = []
    for i in range(len(c0)):
        tokens.append(c0[i])
        tokens.append(c1[2*i]     + 4096)
        tokens.append(c2[4*i]     + 8192)
        tokens.append(c2[4*i+1]   + 12288)
        tokens.append(c1[2*i+1]   + 16384)
        tokens.append(c2[4*i+2]   + 20480)
        tokens.append(c2[4*i+3]   + 24576)
    return tokens

# Locate jsonl and wavs
jsonl_path = None
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == "dataset.jsonl":
            jsonl_path = os.path.join(root, f)
            break
if jsonl_path is None:
    jsonl_path = f"{DRIVE}/dataset.jsonl"

wavs_path = None
for root, dirs, files in os.walk(WORK_DIR):
    for d in dirs:
        if d == "wavs":
            wavs_path = os.path.join(root, d)
            break
    if wavs_path:
        break
if wavs_path is None:
    wavs_path = WORK_DIR

print(f"jsonl: {jsonl_path}")
print(f"wavs:  {wavs_path}")

with open(jsonl_path) as f:
    samples = [json.loads(l) for l in f if l.strip()]

tokenized, skipped = [], 0
for item in tqdm(samples, desc="SNAC encoding"):
    # FIX: Robustly handle Windows paths in the JSONL to get just the filename
    raw_path = item["audio_path"].replace("\\", "/")
    filename = os.path.basename(raw_path)
    wav_path = os.path.join(wavs_path, filename)

    if not os.path.exists(wav_path):
        skipped += 1
        continue
    try:
        tokens = audio_to_tokens(wav_path)
        if tokens is None or len(tokens) < 14:
            skipped += 1
            continue
        tokenized.append({"text": item["text"], "tokens": tokens})
    except Exception as e:
        print(f"  {filename}: {e}")
        skipped += 1

print(f"Encoded: {len(tokenized)}  |  skipped: {skipped}")

tok_path = f"{DRIVE}/tokenized.jsonl"
with open(tok_path, "w") as f:
    for item in tokenized:
        f.write(json.dumps(item) + "\n")
print("Saved:", tok_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

jsonl: /content/work/dataset/dataset.jsonl
wavs:  /content/work/dataset/wavs


SNAC encoding:   1%|▏         | 2/135 [00:00<00:54,  2.43it/s]

...

  seg_00130.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent 

SNAC encoding:  99%|█████████▊| 133/135 [00:20<00:00,  7.11it/s]

  seg_00132.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent 

SNAC encoding: 100%|██████████| 135/135 [00:20<00:00,  6.50it/s]

  seg_00134.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent 

In [27]:
import json, os
from datasets import Dataset

DRIVE        = "/content/drive/MyDrive/rio"
SPEAKER_NAME = "Aswini"   # MUST match test_tts.py exactly

def format_sample(text, tokens):
    audio_str = "".join(f"<custom_token_{t + 10}>" for t in tokens)
    return (
        f"<|im_start|>user\n"
        f"<custom_token_3>{SPEAKER_NAME}: {text}<|eot_id|><custom_token_4>\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"<custom_token_5>{audio_str}<|im_end|>"
    )

tok_path = f"{DRIVE}/tokenized.jsonl"
if not os.path.exists(tok_path):
    raise FileNotFoundError(f"Tokenized file not found at {tok_path}. Please run the encoding cell first.")

with open(tok_path) as f:
    samples = [json.loads(l) for l in f if l.strip()]

if len(samples) == 0:
    print("Error: No samples found in tokenized.jsonl. Check cell 3f05cf12 output.")
else:
    formatted = [{"text": format_sample(s["text"], s["tokens"])} for s in samples]
    print(f"Total: {len(formatted)} samples")
    print("\nPreview (first 300 chars):")
    print(formatted[0]["text"][:300])
    print(f"\nCustom tokens in sample: {formatted[0]['text'].count('<custom_token_')}")

    split = int(len(formatted) * 0.9)
    train_dataset = Dataset.from_list(formatted[:split])
    val_dataset   = Dataset.from_list(formatted[split:])
    print(f"\nTrain: {len(train_dataset)}  |  Val: {len(val_dataset)}")

Total: 122 samples

Preview (first 300 chars):
<|im_start|>user
<custom_token_3>Aswini: Hello. Good morning, sir. Arun Kumar, sir.<|eot_id|><custom_token_4>
<|im_end|>
<|im_start|>assistant
<custom_token_5><custom_token_1058><custom_token_6099><custom_token_10580><custom_token_13775><custom_token_18363><custom_token_22510><custom_token_24865><cu

Custom tokens in sample: 262

Train: 109  |  Val: 13


In [2]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="canopylabs/orpheus-3b-0.1-ft",
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir="/content/checkpoints",
        num_train_epochs=10,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_steps=50,
        save_steps=200,
        save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        dataset_text_field="text",
        max_seq_length=8192,
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("Done. Final loss above.")


ModuleNotFoundError: No module named 'unsloth'

```markdown
After running the fix above, you can try executing your training script again:
```

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/voice_project/orpheus_lora_v2"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to: {SAVE_PATH}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/voice_project/orpheus_lora_v2/tokenizer_config.json.


LoRA adapter saved to: /content/drive/MyDrive/voice_project/orpheus_lora_v2


In [ ]:
merged_path = "/content/orpheus_merged_v2"
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")

# Then convert to GGUF Q4 (4-bit quantized — runs on CPU, ~2GB file)
GGUF_SAVE_PATH = "/content/drive/MyDrive/voice_project/orpheus_aswini_q4.gguf"
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method="q4_k_m"   # 4-bit, good quality-size tradeoff
)
print(f"GGUF saved: {GGUF_SAVE_PATH}")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/orpheus_merged_v2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 560.62it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:54<00:00, 57.14s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/voice_project/orpheus_aswini_q4.gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9756 (llama-b9756-bin-ubuntu-x64.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/voice_project/orpheus_aswini_q4.gguf_gguf/orpheus-3b-0.1-ft.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model fil

In [ ]:
# 1. Install missing Python 3.10 system components and manual pip
!sudo apt-get update -y
!sudo apt-get install python3.10-venv python3.10-distutils -y
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3.10 get-pip.py

# 2. Install specific compatible library versions for Python 3.10
!python3.10 -m pip install -q transformers==4.33.0 tokenizers==0.13.3 TTS==0.22.0 coqui-trainer

# 3. Rewrite the training script
with open('train.py', 'w') as f:
    f.write("""
import os
from pathlib import Path
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts, XttsAudioConfig, XttsArgs
from TTS.utils.manage import ModelManager

os.environ['COQUI_TOS_AGREED'] = '1'
DATASET_DIR = Path('/content/dataset')
OUT_DIR = Path('/content/xtts_finetuned')
OUT_DIR.mkdir(exist_ok=True)
LANGUAGE = 'en'

def aswini_formatter(root_path, meta_file, **kwargs):
    items = []
    with open(meta_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            cols = line.split('|')
            if len(cols) < 2: continue
            items.append({
                'text': cols[1].strip(),
                'audio_file': str(Path(root_path) / 'wavs' / (cols[0].strip() + '.wav')),
                'speaker_name': 'aswini',
                'root_path': root_path,
            })
    return items

manager = ModelManager()
model_path, config_path, _ = manager.download_model('tts_models/multilingual/multi-dataset/xtts_v2')

config = XttsConfig(
    output_path=str(OUT_DIR),
    model_args=XttsArgs(gpt_use_perceiver_resampler=True),
    audio=XttsAudioConfig(sample_rate=22050, output_sample_rate=24000),
    batch_size=4, eval_batch_size=2, num_loader_workers=2, eval_split_size=0.1,
    languages=[LANGUAGE], epochs=200,
    datasets=[BaseDatasetConfig(formatter='aswini', dataset_name='aswini', path=str(DATASET_DIR), meta_file_train=str(DATASET_DIR / 'metadata.csv'), language=LANGUAGE)],
)

train_samples, eval_samples = load_tts_samples(config.datasets, eval_split=True, formatter=aswini_formatter)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=model_path, eval=False)

for name, param in model.named_parameters():
    if 'gpt' not in name: param.requires_grad = False

trainer = Trainer(TrainerArgs(restore_path=None, skip_train_epoch=False, start_with_eval=True, grad_clip=0.5), config, output_path=str(OUT_DIR), model=model, train_samples=train_samples, eval_samples=eval_samples)
trainer.fit()
print('Training successfully finished!')
""")

# 4. Run the training script
!python3.10 train.py

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driver

In [ ]:
# 1. Ensure the training script is written correctly with the right paths
with open('train_xtts.py', 'w') as f:
    f.write("""
import os
from pathlib import Path
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts, XttsAudioConfig, XttsArgs
from TTS.utils.manage import ModelManager

os.environ['COQUI_TOS_AGREED'] = '1'
# Correct path where files were unzipped
DATASET_DIR = Path('/content/segments/dataset')
OUT_DIR = Path('/content/xtts_finetuned')
OUT_DIR.mkdir(exist_ok=True)

def aswini_formatter(root_path, meta_file, **kwargs):
    items = []
    with open(meta_file, 'r', encoding='utf-8') as f:
        for line in f:
            cols = line.strip().split('|')
            if len(cols) < 2: continue
            items.append({
                'text': cols[1].strip(),
                'audio_file': str(Path(root_path) / 'wavs' / (cols[0].strip() + '.wav')),
                'speaker_name': 'aswini',
                'root_path': root_path,
            })
    return items

manager = ModelManager()
model_path, config_path, _ = manager.download_model('tts_models/multilingual/multi-dataset/xtts_v2')

config = XttsConfig(
    output_path=str(OUT_DIR),
    model_args=XttsArgs(gpt_use_perceiver_resampler=True),
    audio=XttsAudioConfig(sample_rate=22050, output_sample_rate=24000),
    batch_size=2, eval_batch_size=2, num_loader_workers=2, epochs=10,
    datasets=[BaseDatasetConfig(formatter='aswini', dataset_name='aswini', path=str(DATASET_DIR), meta_file_train=str(DATASET_DIR / 'metadata.csv'), language='en')],
)

train_samples, eval_samples = load_tts_samples(config.datasets, eval_split=True, formatter=aswini_formatter)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=model_path, eval=False)

for name, param in model.named_parameters():
    if 'gpt' not in name: param.requires_grad = False

trainer = Trainer(TrainerArgs(), config, output_path=str(OUT_DIR), model=model, train_samples=train_samples, eval_samples=eval_samples)
trainer.fit()
""")

# 2. Run the script using the Python 3.10 binary where dependencies are installed
!python3.10 train_xtts.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.4 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0,<3.9; 0.0.10.3 Requires-Python >=3.6.0,<3.9; 0.0.11 Requires-Python >=3.6.0,<3.9; 0.0.12 Requires-Python >=3.6.0,<3.9; 0.0.13.1 Requires-Python >=3.6.0,<3.9; 0.0.13.2 Requires-Python >=3.6.0,<3.9; 0.0.14.1 Requires-Python >=3.6.0,<3.9; 0.0.15 Requires-Python >=3.6.0,<3.9; 0.0.15.1 Requires-Python >=3.6.0,<3.9; 0.0.9 Requires-Python >=3.6.0,<3.9; 0.0.9.1 Requires-Python >=3.6.0,<3.9; 0.0.9.2 Requires-Python >=3.6.0,<3.9; 0.0.9a10 Requires-Python >=3.6.0,<3.9; 0.0.9a9 Requires-Python >=3.6.0,<3.9; 0.1.0 Requires-Python >=3.6.0,<3.10; 0.1.1 Requires-Python >=3.6.0,<3.10; 0.1.2 Requires-Python >=3.6.0,<3.10; 0.1.3 Requires-Python >=3.6.0,<3.10; 0.10.0 Requires-Python >=3.7.0,<3.11; 0.10.1 Requires-Python >=3.7.0,<3.11; 0.10.2 Requires-Python >=3.7.0,<3.11; 0.11.0 Requires-Python >=3.7.0,<3.11; 0.11.1 Req

# Step 1: Install GPT-SoVITS and Dependencies

In [ ]:
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd GPT-SoVITS
!pip install -r requirements.txt
!pip install ipykernel localtunnel

Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5923, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 5923 (delta 49), reused 16 (delta 16), pack-reused 5839 (from 3)
Receiving objects: 100% (5923/5923), 15.13 MiB | 17.41 MiB/s, done.
Resolving deltas: 100% (3379/3379), done.
/content/GPT-SoVITS
Ignoring onnxruntime: markers 'platform_machine == "aarch64" or platform_machine == "arm64"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 148.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 67.2 MB/s  0:00:00
  I

# Step 2: Download Pre-trained Models

In [ ]:
!mkdir -p GPT_SoVITS/pretrained_models
!mkdir -p tools/asr/models
!mkdir -p tools/uvr5/uvr5_weights

# Download necessary weights using gdown or direct links
!gdown 14vS9TfD8E-K_Yv1qN-D1kX_UoG5e_rG_ -O GPT_SoVITS/pretrained_models/gsv-v2final-pretrained.pth

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=14vS9TfD8E-K_Yv1qN-D1kX_UoG5e_rG_

but Gdown can't. Please check connections and permissions.


# Step 3: Launch the Web Interface
Run the cell below, wait for the public URL to appear, and click it. Use the IP address printed as the password if prompted by localtunnel.

In [ ]:
import os
import subprocess
import time

# 1. Install missing dependencies
print("Installing dependencies...")
!pip install -q pypinyin g2p_en jieba sentence_transformers g2pw opencc-python-reimplemented

# 2. Setup folder structure and download missing models
print("Fixing core models and Roberta weights...")
!mkdir -p GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large
!mkdir -p tools/asr/models

# Download Chinese Roberta (essential for the WebUI to start)
roberta_path = "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
if not os.path.exists(f"{roberta_path}/config.json"):
    !wget -q --show-progress -O {roberta_path}/config.json https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/config.json
    !wget -q --show-progress -O {roberta_path}/pytorch_model.bin https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/pytorch_model.bin
    !wget -q --show-progress -O {roberta_path}/tokenizer.json https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/tokenizer.json

# Fix ASR models
asr_tar = 'tools/asr/models/funasr_onnx.tar.gz'
if not os.path.exists('tools/asr/models/funasr_onnx'):
    print("Downloading ASR weights...")
    !rm -f {asr_tar}
    !wget -c -q --show-progress -O {asr_tar} https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/tools/asr/models/funasr_onnx.tar.gz
    !tar -xzf {asr_tar} -C tools/asr/models/

# Download base GSV weight
if not os.path.exists('GPT_SoVITS/pretrained_models/gsv-v2final-pretrained.pth'):
    !wget -q --show-progress -O GPT_SoVITS/pretrained_models/gsv-v2final-pretrained.pth https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained.pth

# 3. Launch WebUI and Tunnel
print("Launching WebUI...")
os.environ['PYTHONPATH'] = '/content/GPT-SoVITS'
!fuser -k 9874/tcp > /dev/null 2>&1

webui_process = subprocess.Popen(["python", "webui.py", "en"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:9874"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(20)

print("\n--- YOUR ACCESS LINK ---")
link_found = False
for _ in range(50):
    line = tunnel_process.stdout.readline()
    if "trycloudflare.com" in line:
        print(line.strip().split(" ")[-1])
        link_found = True
        break
if not link_found: print("Generating... please wait and check logs.")
print("------------------------\n")

print("Live Startup Logs:")
for _ in range(15):
    log = webui_process.stdout.readline().strip()
    if log: print(f"[WebUI]: {log}")

Installing dependencies...
Fixing core models and Roberta weights...
GPT_SoVITS/pretrain 100%[===================>]     690  --.-KB/s    in 0s      
GPT_SoVITS/pretrain 100%[===================>]   1.22G   209MB/s    in 6.1s    
GPT_SoVITS/pretrain 100%[===================>] 262.66K  --.-KB/s    in 0.006s  

gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now
Launching WebUI...

--- YOUR ACCESS LINK ---
trycloudflare.com...
------------------------

Live Startup Logs:
